# Ingesta de fuentes RSS

Este notebook implementa la ingesta de las fuentes RSS configuradas en
PostgreSQL.

## Objetivos

1. Obtener las fuentes RSS activas desde PostgreSQL.
2. Descargar el contenido original de cada feed RSS.
3. Generar un hash SHA-256 para detectar cambios y evitar duplicados.
4. Almacenar nuevas versiones de los feeds en `rss_raw`.
5. Preservar el contenido original para garantizar trazabilidad y permitir
   futuros reprocesamientos.

In [1]:
import hashlib
import os

import psycopg
import requests
from dotenv import load_dotenv

In [2]:
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

True

In [7]:
def get_database_connection():
    """Crea una conexión con la base de datos PostgreSQL configurada.

    Returns:
        psycopg.Connection: Conexión activa a la base de datos PostgreSQL.
    """
    return psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )

In [8]:
with get_database_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT current_database();")
        database_name = cursor.fetchone()[0]

print(f"Conectado correctamente a la base de datos: {database_name}")

Conectado correctamente a la base de datos: cia_db


## Configuración de fuentes RSS

Las fuentes RSS utilizadas por el asistente se almacenan en la tabla
`rss_sources`. Esto permite administrar las fuentes de forma independiente
del código de ingesta.

Para la prueba de concepto se seleccionaron fuentes relacionadas con
tecnología, ciencia y actividad empresarial, priorizando feeds RSS válidos
y con contenido relevante para el dominio de análisis.

In [9]:
rss_sources = [
    {
        "name": "Infobae - Tecno",
        "category": "technology",
        "feed_url": (
            "https://www.infobae.com/arc/outboundfeeds/"
            "rss/category/tecno/"
        ),
        "website_url": "https://www.infobae.com/tecno/",
    },
    {
        "name": "Clarín - Tecnología",
        "category": "technology",
        "feed_url": "https://www.clarin.com/rss/tecnologia/",
        "website_url": "https://www.clarin.com/tecnologia/",
    },
    {
        "name": "Agencia UNQ - Tecnología",
        "category": "technology",
        "feed_url": "https://agencia.unq.edu.ar/?feed=rss2&cat=15",
        "website_url": "https://agencia.unq.edu.ar/?cat=15",
    },
    {
        "name": "Agencia UNQ - Ciencias",
        "category": "science",
        "feed_url": "https://agencia.unq.edu.ar/?feed=rss2&cat=8",
        "website_url": "https://agencia.unq.edu.ar/?cat=8",
    },
    {
        "name": "Agencia UNQ - Empresas",
        "category": "business",
        "feed_url": "https://agencia.unq.edu.ar/?feed=rss2&cat=482",
        "website_url": "https://agencia.unq.edu.ar/?cat=482",
    },
    {
        "name": "UTN - Revista Tecnología y Ciencia",
        "category": "science",
        "feed_url": (
            "https://rtyc.utn.edu.ar/index.php/rtyc/gateway/"
            "plugin/WebFeedGatewayPlugin/rss2"
        ),
        "website_url": "https://rtyc.utn.edu.ar/index.php/rtyc",
    },
    {
        "name": "Página 12 - Ciencia",
        "category": "science",
        "feed_url": (
            "https://www.pagina12.com.ar/arc/outboundfeeds/"
            "rss/secciones/ciencia/notas"
        ),
        "website_url": "https://www.pagina12.com.ar/ciencia/",
    },
]

In [11]:
def insert_rss_sources(sources):
    """Inserta las fuentes RSS configuradas en PostgreSQL.

    Si una fuente ya existe según su URL RSS, no se vuelve a insertar.

    Args:
        sources (list[dict]): Lista de fuentes RSS a registrar.

    Returns:
        int: Cantidad de nuevas fuentes insertadas.
    """
    insert_query = """
        INSERT INTO rss_sources (
            name,
            category,
            feed_url,
            website_url
        )
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (feed_url) DO NOTHING;
    """

    inserted_count = 0

    with get_database_connection() as connection:
        with connection.cursor() as cursor:
            for source in sources:
                cursor.execute(
                    insert_query,
                    (
                        source["name"],
                        source["category"],
                        source["feed_url"],
                        source["website_url"],
                    ),
                )

                inserted_count += cursor.rowcount

    return inserted_count

In [12]:
inserted_sources = insert_rss_sources(rss_sources)

print(f"Fuentes RSS insertadas: {inserted_sources}")

Fuentes RSS insertadas: 7


In [13]:
def get_rss_sources():
    """Obtiene las fuentes RSS registradas en PostgreSQL.

    Returns:
        list[tuple]: Fuentes RSS almacenadas en la base de datos.
    """
    select_query = """
        SELECT
            rss_source_id,
            name,
            category,
            feed_url,
            website_url,
            is_active
        FROM rss_sources
        ORDER BY rss_source_id;
    """

    with get_database_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(select_query)
            return cursor.fetchall()

## Ingesta de contenido RSS

En esta etapa se consultan las fuentes activas almacenadas en `rss_sources`
y se descarga el contenido original proporcionado por cada feed.

El contenido se almacena sin transformaciones en `rss_raw`, permitiendo
conservar una copia de los datos obtenidos desde la fuente original.

Para identificar cambios entre ejecuciones se calcula un hash SHA-256 sobre
el contenido descargado. Si una versión del feed ya fue almacenada para una
fuente determinada, no se vuelve a insertar.

In [15]:
def get_active_rss_sources():
    """Obtiene las fuentes RSS activas configuradas para la ingesta.

    Returns:
        list[tuple]: Fuentes RSS activas almacenadas en PostgreSQL.
    """
    select_query = """
        SELECT
            rss_source_id,
            name,
            category,
            feed_url
        FROM rss_sources
        WHERE is_active = TRUE
        ORDER BY rss_source_id;
    """

    with get_database_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(select_query)
            return cursor.fetchall()

In [16]:
active_sources = get_active_rss_sources()

print(f"Fuentes RSS activas: {len(active_sources)}")

Fuentes RSS activas: 7


### Descarga de fuentes RSS

Antes de persistir los datos en `rss_raw`, se descarga el contenido de cada
fuente activa y se verifican algunos atributos básicos de la respuesta.

En esta etapa no se aplican filtros por palabras clave ni transformaciones
sobre el contenido. El objetivo es validar la adquisición de los datos
originales antes de almacenarlos.

In [17]:
def download_rss_feed(source):
    """Descarga el contenido original de una fuente RSS.

    Args:
        source (tuple): Registro de una fuente RSS activa compuesto por
            identificador, nombre, categoría y URL del feed.

    Returns:
        dict: Resultado de la descarga junto con sus metadatos principales.
    """
    rss_source_id, name, category, feed_url = source

    try:
        response = requests.get(
            feed_url,
            timeout=30,
            headers={"User-Agent": "Mozilla/5.0"},
        )
        response.raise_for_status()

        content = response.content
        content_hash = hashlib.sha256(content).hexdigest()

        return {
            "rss_source_id": rss_source_id,
            "name": name,
            "category": category,
            "feed_url": feed_url,
            "http_status": response.status_code,
            "content_type": response.headers.get("Content-Type"),
            "content_size": len(content),
            "content_hash": content_hash,
            "content": content,
            "success": True,
            "error": None,
        }

    except requests.RequestException as error:
        return {
            "rss_source_id": rss_source_id,
            "name": name,
            "category": category,
            "feed_url": feed_url,
            "http_status": None,
            "content_type": None,
            "content_size": None,
            "content_hash": None,
            "content": None,
            "success": False,
            "error": str(error),
        }

In [18]:
download_results = [
    download_rss_feed(source)
    for source in active_sources
]

In [19]:
df_download_results = pd.DataFrame(download_results).drop(
    columns=["content"]
)

df_download_results

,rss_source_id,name,category,feed_url,http_status,content_type,content_size,content_hash,success,error
0,1,Infobae - Tecno,technology,https://www.infobae.com/arc/outboundfeeds/rss/...,200,application/xml; charset=utf-8,848584,2be712d1eb7e6d0e18711f7f6275b3fdcc56c5917d3c24...,True,None
1,2,Clarín - Tecnología,technology,https://www.clarin.com/rss/tecnologia/,200,text/xml; charset=utf-8,10967,33950a12ca015d8d2250846f7520383f7014438d7b21be...,True,None
2,3,Agencia UNQ - Tecnología,technology,https://agencia.unq.edu.ar/?feed=rss2&cat=15,200,application/rss+xml; charset=UTF-8,220071,1da03a2246ee21e6c46525c435e0d9289e3d0b0d8416ec...,True,None
3,4,Agencia UNQ - Ciencias,science,https://agencia.unq.edu.ar/?feed=rss2&cat=8,200,application/rss+xml; charset=UTF-8,205267,1fd20fcd7e4abb13cab79b43e64ff66e704ac7ccec44fe...,True,None
4,5,Agencia UNQ - Empresas,business,https://agencia.unq.edu.ar/?feed=rss2&cat=482,200,application/rss+xml; charset=UTF-8,310592,398cc927f033673dae491c53d25a4dd9e9f0e44bde5be5...,True,None
5,6,UTN - Revista Tecnología y Ciencia,science,https://rtyc.utn.edu.ar/index.php/rtyc/gateway...,200,text/html; charset=utf-8,34853,de3aefcaf4c459432ae5e838c6037049ef0e5fb19b9c36...,True,None
6,7,Página 12 - Ciencia,science,https://www.pagina12.com.ar/arc/outboundfeeds/...,200,application/xml; charset=utf-8,27627,e6f51c999ccff52473ff823d762c77430d0605f9f39b2f...,True,None


### Persistencia del contenido RAW

Los contenidos descargados correctamente se almacenan en `rss_raw` sin
aplicar transformaciones ni filtros.

Antes de insertar un registro se verifica si el mismo contenido ya fue
almacenado para la fuente correspondiente mediante su hash SHA-256. Esto
permite evitar duplicados entre ejecuciones sucesivas de la ingesta.

Los nuevos registros se almacenan inicialmente con `is_processed = FALSE`
para indicar que todavía no fueron procesados.

In [20]:
def insert_rss_raw(downloads):
    """Almacena en PostgreSQL los contenidos RSS descargados.

    Solo se consideran descargas exitosas. Si el contenido ya existe para
    la misma fuente según su hash SHA-256, el registro no se vuelve a
    insertar.

    Args:
        downloads (list[dict]): Resultados obtenidos durante la descarga
            de las fuentes RSS.

    Returns:
        int: Cantidad de nuevos registros insertados.
    """
    insert_query = """
        INSERT INTO rss_raw (
            fk_rss_source_id,
            raw_content,
            content_hash,
            retrieved_at,
            is_processed
        )
        VALUES (
            %s,
            %s,
            %s,
            CURRENT_TIMESTAMP,
            FALSE
        )
        ON CONFLICT (fk_rss_source_id, content_hash)
        DO NOTHING;
    """

    inserted_count = 0

    with get_database_connection() as connection:
        with connection.cursor() as cursor:
            for download in downloads:
                if not download["success"]:
                    continue

                raw_content = download["content"].decode(
                    "utf-8",
                    errors="replace",
                )

                cursor.execute(
                    insert_query,
                    (
                        download["rss_source_id"],
                        raw_content,
                        download["content_hash"],
                    ),
                )

                inserted_count += cursor.rowcount

    return inserted_count

In [21]:
inserted_raw = insert_rss_raw(download_results)

print(f"Registros RAW insertados: {inserted_raw}")

Registros RAW insertados: 7
